In [ ]:
from pathlib import Path

import numpy as np
import zarr
from ls_mcmc import logging, sampling

from cardiac_electrophysiology import mcmc_builder, posterior_builder
from cardiac_electrophysiology.ls_bip import laplace
from cardiac_electrophysiology.utils import visualization

In [ ]:
posterior_settings = posterior_builder.PosteriorBuilderSettings(
    paths=posterior_builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_mcmc_logfile.log"),
    ),
    prior_parameters=posterior_builder.PriorParameters(
        kappa=0.05,
        tau=10,
        seed=0,
    ),
    eikonal_parameters=posterior_builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=1.5,
        transversal_velocity=1,
    ),
    observation_parameters=posterior_builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-3,
        seed=0,
    ),
    logger_settings=posterior_builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)
builder = posterior_builder.PosteriorBuilder(posterior_settings)
posterior, additional_output = builder.build(return_additional_data=True)

In [ ]:
optimization_data = np.load("../results/optimization_data.npz")
lowrank_data = np.load("../results/lowrank_data.npz")

map_estimate = optimization_data["map_estimate"]
laplace_approximation = laplace.LaplaceApproximation(
    map_estimate=map_estimate,
    hessian_eigenvalues=lowrank_data["hessian_eigenvalues"],
    hessian_eigenvectors=lowrank_data["hessian_eigenvectors"],
    prior_covariance_eigenvalues=lowrank_data["prior_covariance_eigenvalues"],
    prior_covariance_eigenvectors=lowrank_data["prior_covariance_eigenvectors"],
    prior=posterior.prior,
)
builder_settings = mcmc_builder.MCMCBuilderSettings(
    mcmc_model_settings=mcmc_builder.MCMCModelSettings(
        log_posterior=posterior,
        laplace_approximation=laplace_approximation,
        reference_point=map_estimate,
        step_width=5e-2,
        index_to_track=42,
    ),
    logging_settings=logging.LoggerSettings(
        do_printing=False,
        logfile_path=Path("../results/lsmcmc_logfile.log"),
    ),
    storage_path=Path("../results/mcmc_samples.zarr"),
    storage_chunk_size=10,
    overwrite_existing_storage=True,
)
builder = mcmc_builder.MCMCBuilder(builder_settings)
mcmc_sampler = builder.build()

In [ ]:
initial_state = np.load("initial_state.npy")
sampler_settings=sampling.SamplerRunSettings(
    num_samples=10000,
    initial_state=initial_state,
    print_interval=1,
    checkpoint_path=None,
)
storage, outputs = mcmc_sampler.run(sampler_settings)

In [ ]:
samples_1 = zarr.load("../results/mcmc_samples/06_mcmc_samples.zarr/data")
samples_2 = zarr.load("../results/mcmc_samples/07_mcmc_samples.zarr/data")
samples = np.concatenate((samples_1, samples_2), axis=0)
mcmc_mean = np.mean(samples, axis=0)
mcmc_var = np.var(samples, axis=0)

In [ ]:
visualization.visualize_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=mcmc_mean,
    circular=False,
)